# 01 · Data and preprocessing
**Question:** How do we turn a judgment into reliable training examples?

**Input:** OpenNyAI judgments with sentence spans and 13 rhetorical roles. **Output:** one ordered row per sentence, official splits, and an audit report.
This notebook has a small invented demo for tomorrow's presentation. Demo scores are not project results. Set `DEMO=False` to prepare the real data.

## Setup — run once
The project uses an isolated Python environment so its Hugging Face packages do not conflict with Colab's installed Gradio/Diffusers.
`%%project` means “run this ordinary Python cell in that environment.” Variables persist between these cells.
The first setup downloads dependencies and can take several minutes. No token is needed for the demo or downloaded trained model.
For the gated role dataset, accept its access conditions and add `HF_TOKEN` in Colab Secrets; never paste it into a cell.

In [ ]:
from pathlib import Path
import os, subprocess, sys
ROOT = Path(os.environ.get('PROJECT_ROOT', '/content/legal-judgment-summarizer' if Path('/content').exists() else Path.cwd()))
if not (ROOT / 'src').exists() and (ROOT.parent / 'src').exists():
    ROOT = ROOT.parent
if not (ROOT / 'src').exists():
    subprocess.check_call(['git', 'clone', 'https://github.com/Mananfrfr/legal-judgment-summarizer.git', str(ROOT)])
sys.path.insert(0, str(ROOT / 'src'))
from notebook_runtime import setup
setup(ROOT)


In [ ]:
%%project
from pathlib import Path
import os, json
import pandas as pd
from IPython.display import display, Markdown
ROOT = Path(os.environ['PROJECT_ROOT'])


## 1. Load the data
OpenNyAI teaches sentence roles (facts, arguments, reasoning, rulings). IN-Abs is a separate dataset of judgments paired with summaries, used later for sentence selection. They solve different tasks.
The role dataset is pinned to the revision used by the recorded experiments. We keep the official document splits; random sentence splitting could leak the same case into training and validation.

In [ ]:
%%project
from data_utils import demo_role_data, load_role_data
DEMO = True
if DEMO:
    frames, audit = demo_role_data()
else:
    frames, audit = load_role_data(ROOT / 'data/raw/roles', download=True)
print('SYNTHETIC TEACHING DEMO' if DEMO else 'OFFICIAL ROLE DATASET')
display(pd.DataFrame({name: {'documents': df.doc_index.nunique(), 'sentences': len(df)} for name, df in frames.items()}).T)


## 2. Inspect one training example
A sentence has text, a label, its judgment ID, and character offsets. We preserve legal wording, numbers and negation. Stop-word removal is a feature choice for some TF-IDF experiments, not a destructive edit to the source.

In [ ]:
%%project
train = frames['train']
display(train[['doc_index', 'sentence_index', 'start', 'end', 'text', 'label']].head(8))
display(train.label.value_counts().rename('training_sentences').to_frame())


## 3. Understand the annotation policy
Some documents have more than one annotator. Merging their rows would duplicate sentences or attach contradictory labels. We use an explicitly adjudicated annotation when present; otherwise the first nonempty set, and record disagreements. We sort by offsets and reject duplicate spans, unknown labels and overlapping document splits.
The implementation is in `src/annotations.py`; the short public interface is `prepare_frames` in `src/data_utils.py`.

In [ ]:
%%project
audit_summary = {name: {key: value for key, value in info.items() if key != 'multi_annotation_details'}
                 for name, info in audit.items() if isinstance(info, dict)}
display(pd.DataFrame(audit_summary).T)
print({key: value for key, value in audit.items() if 'duplicate_documents' in key})


## 4. Save the prepared splits
Notebook 02 reads these files. The demo and official data go to different folders, so an invented example cannot silently replace the real dataset.

In [ ]:
%%project
from data_utils import save_frames
prepared = ROOT / 'data/processed' / ('demo' if DEMO else 'official')
save_frames(frames, audit, prepared)
print('Saved:', prepared)


## 5. Separate summarization data
The IN-Abs experiment used **1,000 train, 40 development and 100 official test cases**, with a 400-word summary budget and 120 sampled candidate sentences. The saved train/dev IDs are versioned. The official test has already been evaluated; do not repeatedly tune against it.
Download the official ZIP with `scripts/download_data.py inabs`; its MD5 is checked. Dataset URLs, licenses/access conditions and citations are in `data/README.md`.

In [ ]:
%%project
split_ids = json.loads((ROOT / 'data/splits.json').read_text())
print({key: len(value) for key, value in split_ids.items() if key.endswith('_ids')})


### Explain it to your faculty
“I preserve sentence order and judgment boundaries, audit labels, and keep training and evaluation cases separate. OpenNyAI provides role labels; IN-Abs provides reference summaries. I never use a test summary to construct training labels.”

**Next:** 02 trains and compares models. **Ask yourself:** Why would splitting individual sentences randomly give an overly optimistic score?